In [1]:
# packages
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import json
import os

In [2]:
SAMPLE_PATH = Path("/kaggle/input/datasets/lianestrauch/bert-base-samples")

In [3]:
# create function: get_kdistances(file, identifier)

def get_kdistance(
    sample_file: str,
    identifier: str,
    k_values: list[int],
): #-> dict[int, float]:

    if not sample_file.is_file():
        raise FileNotFoundError(
            f"Sample file does not exist: {sample_file}"
        )

    if sample_file.suffix.lower() != ".parquet":
        raise ValueError(
            f"Expected a parquet file, got: {sample_file}"
        )

    sample = pd.read_parquet(sample_file)

    col_name = sample.columns[-1]

    # create output file 
    k_distances_dict = {}

    X = np.vstack(sample[col_name].values)

    for k in k_values:
        neighbors = NearestNeighbors(
            n_neighbors=k,
            metric="cosine"
        )
    
        neighbors.fit(X)
    
        distances, _ = neighbors.kneighbors(X)
    
        k_distances = np.sort(distances[:, k - 1])[::-1]

        # sort distances and ids together
        sorted_indices = np.argsort(k_distances)
        sorted_distances = k_distances[sorted_indices]
        sorted_ids = sample["id"].values[sorted_indices]

        # Rank after sorting
        ranks = np.arange(len(sorted_distances))

        # Create export DataFrame
        k_distance_df = pd.DataFrame({
        "rank": ranks,
        "id": sorted_ids,
        "k_distance": sorted_distances
        })

        csv_path = f"k_distance_{identifier}_{k}.csv"
        
        k_distance_df.to_csv(
        csv_path,
        index=False
        )
        
        print(f"  Saved: {csv_path}")

    return

#get_kdistance(file = "sample_bert_base_last.parquet", identifier = "bb_ll", k_values = [50, 100])
    
    

In [4]:
all_samples = sorted(SAMPLE_PATH.glob("*.parquet"))
all_samples

all_identifier = [str(sample)[71:] for sample in all_samples]
all_identifier = [sample[:-8] for sample in all_identifier]
all_identifier

['2nd_last', 'last', 'mean_last4']

In [5]:
for file, name in zip(all_samples, all_identifier):
    get_kdistance(sample_file = file,
             identifier = f"bb_{name}", #bert-base = bb
             k_values = [5, 10, 50, 100] #on full: 50, 100, 500, 1000
            )

  Saved: k_distance_bb_2nd_last_5.csv
  Saved: k_distance_bb_2nd_last_10.csv
  Saved: k_distance_bb_2nd_last_50.csv
  Saved: k_distance_bb_2nd_last_100.csv
  Saved: k_distance_bb_last_5.csv
  Saved: k_distance_bb_last_10.csv
  Saved: k_distance_bb_last_50.csv
  Saved: k_distance_bb_last_100.csv
  Saved: k_distance_bb_mean_last4_5.csv
  Saved: k_distance_bb_mean_last4_10.csv
  Saved: k_distance_bb_mean_last4_50.csv
  Saved: k_distance_bb_mean_last4_100.csv
